In [ ]:
# =============================================================
#  PRICER D'OPTIONS — Black-Scholes & Monte Carlo
#  Projet Finance Quantitative
# =============================================================
#
#  Ce fichier calcule le prix d'une option financière
#  de deux façons différentes :
#    1. La formule de Black-Scholes (calcul direct, exact)
#    2. La simulation Monte Carlo (on simule des milliers
#       de scénarios possibles et on fait la moyenne)
#
#  Une option, c'est quoi ?
#  -> Un CALL donne le droit d'ACHETER une action à un prix fixé à l'avance (le strike K)
#  -> Un PUT  donne le droit de VENDRE  une action à un prix fixé à l'avance (le strike K)
# =============================================================


# --- On importe les outils dont on a besoin ---
import numpy as np                   # Pour les maths et les tableaux de nombres
import pandas as pd                  # Pour afficher des tableaux propres
import matplotlib.pyplot as plt      # Pour faire des graphiques
from scipy.stats import norm         # Pour la loi normale (courbe en cloche)
from scipy.optimize import brentq    # Pour résoudre des équations numériquement
import warnings
warnings.filterwarnings('ignore')    # On cache les avertissements inutiles

# Style sombre pour les graphiques (comme GitHub)
plt.rcParams.update({
    'figure.facecolor': '#0d1117',
    'axes.facecolor':   '#161b22',
    'axes.edgecolor':   '#30363d',
    'axes.labelcolor':  '#c9d1d9',
    'xtick.color':      '#8b949e',
    'ytick.color':      '#8b949e',
    'text.color':       '#c9d1d9',
    'grid.color':       '#21262d',
    'grid.linestyle':   '--',
    'figure.dpi':       120
})

# On fixe la graine aléatoire pour avoir les mêmes résultats à chaque exécution
np.random.seed(42)


# =============================================================
#  PARTIE 1 — LES PARAMÈTRES DE NOTRE OPTION
# =============================================================
#
#  Imaginons qu'on veut pricer une option sur une action qui
#  vaut 100€ aujourd'hui, avec un strike à 105€ dans 1 an.

S0    = 100.0   # Prix actuel de l'action (le "spot")
K     = 105.0   # Prix auquel on pourra acheter/vendre (le "strike")
T     = 1.0     # Dans combien de temps expire l'option (en années)
r     = 0.05    # Taux d'intérêt sans risque (ici 5% par an)
sigma = 0.20    # Volatilité de l'action (ici 20% — mesure de l'agitation du prix)
q     = 0.00    # Dividendes versés par l'action (ici 0)

# Affichage des paramètres
print("--- Paramètres de l'option ---")
print(f"  Prix actuel de l'action  : {S0} €")
print(f"  Prix d'exercice (strike) : {K} €")
print(f"  Durée jusqu'à maturité   : {T} an(s)")
print(f"  Taux sans risque         : {r*100}%")
print(f"  Volatilité               : {sigma*100}%")
print(f"  Dividendes               : {q*100}%")


# =============================================================
#  PARTIE 2 — FORMULE DE BLACK-SCHOLES
# =============================================================
#
#  Black-Scholes est une formule mathématique qui donne
#  directement le prix d'une option. C'est comme une recette :
#  tu mets les ingrédients (S, K, T, r, sigma) et tu obtiens
#  le prix exact (sous certaines hypothèses simplificatrices).
#
#  La formule utilise d1 et d2, deux nombres intermédiaires,
#  et la fonction N() qui est la loi normale cumulative
#  (la probabilité que quelque chose arrive).

def black_scholes(S, K, T, r, sigma, q=0.0, option_type='call'):
    """
    Calcule le prix d'une option européenne avec Black-Scholes.

    On calcule aussi les "Greeks" : ce sont des mesures de
    sensibilité qui disent comment le prix de l'option change
    quand les paramètres bougent.

    - Delta : comment le prix change si l'action monte de 1€
    - Gamma : comment le Delta lui-même change
    - Vega  : comment le prix change si la volatilité monte de 1%
    - Theta : combien l'option perd de valeur chaque jour
    - Rho   : comment le prix change si les taux bougent de 1%
    """

    # Cas spécial : si l'option est déjà expirée
    if T <= 0:
        if option_type == 'call':
            return max(S - K, 0), {}   # On gagne S-K si positif, sinon 0
        else:
            return max(K - S, 0), {}

    # Calcul des deux quantités intermédiaires d1 et d2
    # d1 intègre le prix actuel, le strike, le temps, le taux et la volatilité
    d1 = (np.log(S / K) + (r - q + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)

    # Calcul du prix selon le type d'option
    if option_type == 'call':
        # Prix du call : on paie le droit d'acheter
        price = S * np.exp(-q * T) * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)
        delta = np.exp(-q * T) * norm.cdf(d1)
        rho   = K * T * np.exp(-r * T) * norm.cdf(d2) / 100
    else:
        # Prix du put : on paie le droit de vendre
        price = K * np.exp(-r * T) * norm.cdf(-d2) - S * np.exp(-q * T) * norm.cdf(-d1)
        delta = -np.exp(-q * T) * norm.cdf(-d1)
        rho   = -K * T * np.exp(-r * T) * norm.cdf(-d2) / 100

    # Les Greeks communs aux deux types d'options
    gamma = np.exp(-q * T) * norm.pdf(d1) / (S * sigma * np.sqrt(T))
    vega  = S * np.exp(-q * T) * norm.pdf(d1) * np.sqrt(T) / 100
    theta = (
        -(S * np.exp(-q * T) * norm.pdf(d1) * sigma) / (2 * np.sqrt(T))
        - r * K * np.exp(-r * T) * norm.cdf(d2 if option_type == 'call' else -d2)
        + q * S * np.exp(-q * T) * norm.cdf(d1 if option_type == 'call' else -d1)
    ) / 365   # Divisé par 365 pour avoir la perte par jour

    greeks = {
        'Delta': delta,
        'Gamma': gamma,
        'Vega' : vega,
        'Theta': theta,
        'Rho'  : rho,
    }

    return price, greeks


# On calcule les prix pour le call et le put
bs_call, greeks_call = black_scholes(S0, K, T, r, sigma, q, option_type='call')
bs_put,  greeks_put  = black_scholes(S0, K, T, r, sigma, q, option_type='put')

print("\n--- Résultats Black-Scholes ---")
print(f"  Prix du CALL : {bs_call:.4f} €")
print(f"  Prix du PUT  : {bs_put:.4f} €")

print("\n  Greeks du CALL :")
for nom, valeur in greeks_call.items():
    print(f"    {nom:6s} = {valeur:.4f}")

# Vérification de la parité put-call
# C'est une relation mathématique qui doit toujours être vraie :
# Prix(Call) - Prix(Put) = Prix action - Strike actualisé
parite_gauche = bs_call - bs_put
parite_droite = S0 * np.exp(-q * T) - K * np.exp(-r * T)
print(f"\n  Vérification parité put-call :")
print(f"    C - P           = {parite_gauche:.6f}")
print(f"    S - K*e^(-rT)   = {parite_droite:.6f}")
print(f"    Égaux ? {'✅ Oui' if abs(parite_gauche - parite_droite) < 1e-10 else '❌ Non'}")


# =============================================================
#  PARTIE 3 — SIMULATION MONTE CARLO
# =============================================================
#
#  L'idée est simple : au lieu d'une formule, on simule
#  des milliers de futurs possibles pour le prix de l'action.
#
#  Pour chaque scénario :
#    1. On tire un nombre aléatoire (Z) qui représente
#       le mouvement du marché
#    2. On calcule le prix final de l'action (ST)
#    3. On calcule combien vaut l'option dans ce scénario
#
#  Ensuite, on fait la moyenne de tous les scénarios.
#  Avec assez de simulations, on approche le vrai prix.
#
#  Astuce "variables antithétiques" :
#  Pour chaque tirage aléatoire Z, on utilise aussi -Z.
#  Ca réduit l'erreur sans coût supplémentaire important.

def monte_carlo_pricer(S, K, T, r, sigma, q=0.0, option_type='call',
                       n_simulations=100_000, antithetic=True, seed=42):
    """
    Estime le prix d'une option par simulation Monte Carlo.

    On retourne :
    - price : le prix estimé
    - se    : l'erreur standard (à quel point on est précis)
    - ci    : l'intervalle de confiance à 95%
              (on est sûr à 95% que le vrai prix est dans cet intervalle)
    """

    rng = np.random.default_rng(seed)   # Générateur de nombres aléatoires

    # Tirage des nombres aléatoires
    if antithetic:
        # On tire N/2 nombres, puis on ajoute leurs opposés
        Z = rng.standard_normal(n_simulations // 2)
        Z = np.concatenate([Z, -Z])     # Doubles les scénarios symétriquement
    else:
        Z = rng.standard_normal(n_simulations)

    # Prix final de l'action dans chaque scénario
    # C'est la formule exacte du mouvement brownien géométrique
    ST = S * np.exp((r - q - 0.5 * sigma**2) * T + sigma * np.sqrt(T) * Z)

    # Combien vaut l'option à la fin dans chaque scénario ?
    if option_type == 'call':
        # Le call vaut max(ST - K, 0) : on gagne si l'action monte au-dessus du strike
        payoffs = np.maximum(ST - K, 0)
    else:
        # Le put vaut max(K - ST, 0) : on gagne si l'action descend sous le strike
        payoffs = np.maximum(K - ST, 0)

    # On actualise les gains (l'argent de demain vaut moins qu'aujourd'hui)
    gains_actualises = np.exp(-r * T) * payoffs

    # Statistiques finales
    price = gains_actualises.mean()                             # Prix estimé = moyenne
    se    = gains_actualises.std() / np.sqrt(n_simulations)    # Erreur standard
    ci    = (price - 1.96 * se, price + 1.96 * se)             # Intervalle de confiance 95%

    return price, se, ci


# On lance la simulation avec 200 000 scénarios
N = 200_000

mc_call, se_call, ci_call = monte_carlo_pricer(S0, K, T, r, sigma, q, 'call', N)
mc_put,  se_put,  ci_put  = monte_carlo_pricer(S0, K, T, r, sigma, q, 'put',  N)

print(f"\n--- Résultats Monte Carlo (N = {N:,} simulations) ---")
print(f"  CALL : {mc_call:.4f} €  (± {1.96*se_call:.4f} à 95%)")
print(f"  PUT  : {mc_put:.4f}  €  (± {1.96*se_put:.4f} à 95%)")


# =============================================================
#  PARTIE 4 — COMPARAISON DES DEUX MÉTHODES
# =============================================================

print("\n--- Comparaison Black-Scholes vs Monte Carlo ---")

comparaison = pd.DataFrame({
    'Méthode'          : ['Black-Scholes', 'Monte Carlo', 'Écart absolu', 'Écart relatif (%)'],
    'CALL'             : [bs_call, mc_call,
                          abs(bs_call - mc_call),
                          abs(bs_call - mc_call) / bs_call * 100],
    'PUT'              : [bs_put, mc_put,
                          abs(bs_put - mc_put),
                          abs(bs_put - mc_put) / bs_put * 100],
}).set_index('Méthode').round(6)

print(comparaison.to_string())

# Graphique : comment le prix de l'option évolue selon le prix de l'action
fig = plt.figure(figsize=(16, 6))
fig.patch.set_facecolor('#0d1117')

# --- Graphique gauche : Prix Call et Put selon le cours de l'action ---
ax1 = fig.add_subplot(121)

S_vals = np.linspace(60, 150, 200)   # On fait varier le prix de l'action de 60 à 150
prix_calls = [black_scholes(s, K, T, r, sigma, q, 'call')[0] for s in S_vals]
prix_puts  = [black_scholes(s, K, T, r, sigma, q, 'put')[0]  for s in S_vals]

ax1.plot(S_vals, prix_calls, color='#58a6ff', lw=2.5, label='Call')
ax1.plot(S_vals, prix_puts,  color='#f78166', lw=2.5, label='Put')
ax1.axvline(S0, color='#3fb950', linestyle='--', alpha=0.7, label=f'Prix actuel S₀ = {S0}€')
ax1.axvline(K,  color='#e3b341', linestyle=':',  alpha=0.7, label=f'Strike K = {K}€')
ax1.scatter([S0], [bs_call], color='#58a6ff', s=120, zorder=5)  # Notre call
ax1.scatter([S0], [bs_put],  color='#f78166', s=120, zorder=5)  # Notre put
ax1.set_xlabel("Prix de l'action (€)")
ax1.set_ylabel("Prix de l'option (€)")
ax1.set_title("Prix des options selon le cours de l'action", color='white', pad=12)
ax1.legend(framealpha=0.3, facecolor='#21262d', edgecolor='#30363d')

# --- Graphique droite : Surface 3D du prix du Call ---
# Montre comment le prix change selon DEUX variables à la fois
# (le cours de l'action ET la volatilité)
S_range     = np.linspace(70, 140, 50)
sigma_range = np.linspace(0.05, 0.50, 50)
SS, SSIG    = np.meshgrid(S_range, sigma_range)

call_surface = np.array([
    [black_scholes(s, K, T, r, sig, q, 'call')[0] for s in S_range]
    for sig in sigma_range
])

ax2 = fig.add_subplot(122, projection='3d')
ax2.set_facecolor('#161b22')
surf = ax2.plot_surface(SS, SSIG * 100, call_surface, cmap='plasma', alpha=0.9)
ax2.set_xlabel("Cours de l'action")
ax2.set_ylabel('Volatilité (%)')
ax2.set_zlabel('Prix du Call')
ax2.set_title('Surface de prix du Call\n(selon cours et volatilité)', color='white', pad=10)
ax2.xaxis.pane.fill = False
ax2.yaxis.pane.fill = False
ax2.zaxis.pane.fill = False
fig.colorbar(surf, ax=ax2, shrink=0.5)

plt.suptitle('Black-Scholes — Visualisation des prix', color='#c9d1d9', fontsize=14)
plt.tight_layout()
plt.savefig('bs_surface.png', dpi=130, bbox_inches='tight', facecolor='#0d1117')
plt.show()


# =============================================================
#  PARTIE 5 — CONVERGENCE DE MONTE CARLO
# =============================================================
#
#  Question : combien de simulations faut-il pour être précis ?
#
#  On va tester avec 10, 100, 1000, ... jusqu'à 1 million
#  de simulations, et voir comment l'estimation s'approche
#  de la vraie valeur Black-Scholes.
#
#  La théorie dit que l'erreur diminue en 1/√N :
#  -> 100x plus de simulations = 10x plus précis
#  -> Pour diviser l'erreur par 10, il faut 100x plus de calculs

n_valeurs = np.unique(np.round(np.logspace(1, 6, 80)).astype(int))

prix_mc  = []
erreurs  = []

for n in n_valeurs:
    p, se, _ = monte_carlo_pricer(S0, K, T, r, sigma, q, 'call',
                                   n_simulations=n, antithetic=True, seed=42)
    prix_mc.append(p)
    erreurs.append(1.96 * se)   # Demi-largeur de l'intervalle de confiance 95%

prix_mc = np.array(prix_mc)
erreurs = np.array(erreurs)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.patch.set_facecolor('#0d1117')

# --- Graphique gauche : Convergence vers le vrai prix ---
ax = axes[0]
ax.semilogx(n_valeurs, prix_mc, color='#58a6ff', lw=1.5, label='Estimation Monte Carlo')
ax.fill_between(n_valeurs,
                prix_mc - erreurs,
                prix_mc + erreurs,
                alpha=0.25, color='#58a6ff', label='Intervalle de confiance 95%')
ax.axhline(bs_call, color='#f78166', lw=2, linestyle='--',
           label=f'Vraie valeur Black-Scholes = {bs_call:.4f}€')
ax.set_xlabel('Nombre de simulations (échelle log)')
ax.set_ylabel('Prix estimé du Call (€)')
ax.set_title("Convergence de Monte Carlo vers Black-Scholes", color='white', pad=12)
ax.legend(framealpha=0.3, facecolor='#21262d', edgecolor='#30363d')
ax.set_xlim(n_valeurs[0], n_valeurs[-1])

# --- Graphique droite : Réduction de l'erreur ---
ax2 = axes[1]
ax2.loglog(n_valeurs, erreurs, color='#e3b341', lw=2,
           label="Erreur de l'estimation")

# Courbe théorique : l'erreur doit diminuer en 1/√N
constante = erreurs[0] * np.sqrt(n_valeurs[0])
ax2.loglog(n_valeurs, constante / np.sqrt(n_valeurs),
           color='#3fb950', lw=1.5, linestyle=':',
           label='Théorie : erreur en 1/√N')

ax2.set_xlabel('Nombre de simulations (échelle log)')
ax2.set_ylabel("Erreur de l'estimation (échelle log)")
ax2.set_title("L'erreur diminue quand N augmente", color='white', pad=12)
ax2.legend(framealpha=0.3, facecolor='#21262d', edgecolor='#30363d')
ax2.set_xlim(n_valeurs[0], n_valeurs[-1])

plt.suptitle('Monte Carlo — Convergence et précision', color='#c9d1d9', fontsize=14)
plt.tight_layout()
plt.savefig('mc_convergence.png', dpi=130, bbox_inches='tight', facecolor='#0d1117')
plt.show()

# Résultat avec 1 million de simulations (très précis)
print("\nTest avec N = 1 000 000 simulations :")
p_1m, se_1m, ci_1m = monte_carlo_pricer(S0, K, T, r, sigma, q, 'call', 1_000_000)
print(f"  Estimation MC   = {p_1m:.6f} €")
print(f"  Vraie valeur BS = {bs_call:.6f} €")
print(f"  Erreur          = {abs(p_1m - bs_call):.6f} €")
print(f"  IC 95%          = [{ci_1m[0]:.6f}, {ci_1m[1]:.6f}]")


# =============================================================
#  PARTIE 6 (BONUS) — VOLATILITÉ IMPLICITE
# =============================================================
#
#  Problème inverse : en pratique, on observe le PRIX de l'option
#  sur le marché, mais on ne connaît pas la "vraie" volatilité.
#
#  La volatilité implicite, c'est la volatilité qu'il faudrait
#  mettre dans Black-Scholes pour retrouver le prix observé.
#
#  On résout l'équation : Black-Scholes(sigma) = Prix_marché
#  Avec la méthode de Brent (une sorte de dichotomie avancée).
#
#  Smile de volatilité :
#  En réalité, la vol. implicite n'est pas la même pour tous les strikes.
#  Ce phénomène s'appelle le "smile" (sourire) de volatilité,
#  et montre que le modèle BS est une simplification de la réalité.

def implied_volatility(prix_marche, S, K, T, r, q=0.0, option_type='call'):
    """
    Retrouve la volatilité implicite à partir du prix de marché.

    On cherche sigma tel que :
        Black_Scholes(sigma) = prix_marche

    Retourne NaN si aucune solution trouvée.
    """

    # La fonction qu'on veut annuler : BS(sigma) - Prix_marché = 0
    def ecart(sigma):
        prix_bs, _ = black_scholes(S, K, T, r, sigma, q, option_type)
        return prix_bs - prix_marche

    # On cherche entre sigma = 0.01% et sigma = 500%
    try:
        sigma_impl = brentq(ecart, 1e-4, 5.0, xtol=1e-8)
        return sigma_impl
    except ValueError:
        return np.nan   # Pas de solution trouvée


# Vérification : si on donne à BS le prix qu'il a calculé,
# on doit retrouver la volatilité de départ
iv_call = implied_volatility(bs_call, S0, K, T, r, q, 'call')
iv_put  = implied_volatility(bs_put,  S0, K, T, r, q, 'put')

print("\n--- Volatilité implicite ---")
print(f"  Volatilité d'origine       : {sigma*100:.2f}%")
print(f"  Vol. implicite retrouvée (Call) : {iv_call*100:.4f}%")
print(f"  Vol. implicite retrouvée (Put)  : {iv_put*100:.4f}%")
print("  -> On retrouve bien la même valeur ✅")

# Construction du smile de volatilité
# On simule des prix de marché avec une vraie vol qui varie selon le strike
strikes = np.linspace(70, 140, 30)

# Volatilité "réelle" du marché (plus élevée pour les strikes extrêmes)
log_moneyness = np.log(strikes / S0)
sigma_reel     = 0.20 + 0.05 * log_moneyness**2 - 0.02 * log_moneyness

# Prix simulés avec ces volatilités réelles
prix_marche_simules = [black_scholes(S0, k, T, r, sig, q, 'call')[0]
                       for k, sig in zip(strikes, sigma_reel)]

# Vol. implicites extraites de ces prix
vols_implicites = [implied_volatility(p, S0, k, T, r, q, 'call')
                   for p, k in zip(prix_marche_simules, strikes)]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.patch.set_facecolor('#0d1117')

# --- Graphique gauche : Smile de volatilité ---
ax1 = axes[0]
moneyness = strikes / S0   # Rapport Strike / Prix_actuel (1 = "à la monnaie")
ax1.plot(moneyness, np.array(vols_implicites) * 100,
         'o-', color='#58a6ff', lw=2.5, ms=5, label='Vol. implicite observée')
ax1.axhline(sigma * 100, color='#f78166', linestyle='--', lw=1.5,
            label=f'Vol. constante BS = {sigma*100:.0f}%')
ax1.axvline(1.0, color='#3fb950', linestyle=':', alpha=0.7,
            label='À la monnaie (K = S₀)')
ax1.set_xlabel('Moneyness  K / S₀\n(< 1 = out of the money, > 1 = in the money)')
ax1.set_ylabel('Volatilité implicite (%)')
ax1.set_title("Smile de volatilité\n(la vol. réelle varie selon le strike)", color='white', pad=10)
ax1.legend(framealpha=0.3, facecolor='#21262d', edgecolor='#30363d')

# --- Graphique droite : Structure par terme ---
# La vol. implicite change aussi selon la maturité de l'option
maturites = np.linspace(0.05, 3.0, 50)
sigma_terme = 0.15 + 0.08 * np.exp(-maturites) + 0.02 * maturites * np.exp(-0.5 * maturites)

ax2 = axes[1]
ax2.plot(maturites * 12, sigma_terme * 100, color='#e3b341', lw=2.5)
ax2.fill_between(maturites * 12, sigma_terme * 100, alpha=0.15, color='#e3b341')
ax2.axvline(T * 12, color='#f78166', linestyle='--', lw=1.5,
            label=f'Notre option (T = {int(T*12)} mois)')
ax2.set_xlabel('Maturité de l\'option (mois)')
ax2.set_ylabel('Volatilité implicite ATM (%)')
ax2.set_title("Structure par terme de la volatilité\n(la vol. change aussi avec la durée)",
              color='white', pad=10)
ax2.legend(framealpha=0.3, facecolor='#21262d', edgecolor='#30363d')

plt.suptitle("Volatilité Implicite — Smile & Structure par terme", color='#c9d1d9', fontsize=14)
plt.tight_layout()
plt.savefig('iv_smile.png', dpi=130, bbox_inches='tight', facecolor='#0d1117')
plt.show()


# =============================================================
#  RÉCAPITULATIF FINAL
# =============================================================

print("\n" + "="*55)
print("  RÉCAPITULATIF")
print("="*55)
print(f"\n  CALL")
print(f"    Black-Scholes (exact)  : {bs_call:.4f} €")
print(f"    Monte Carlo (N={N:,}) : {mc_call:.4f} €")
print(f"    Écart                  : {abs(bs_call - mc_call):.6f} €  ({abs(bs_call - mc_call)/bs_call*100:.4f}%)")
print(f"\n  PUT")
print(f"    Black-Scholes (exact)  : {bs_put:.4f} €")
print(f"    Monte Carlo (N={N:,}) : {mc_put:.4f} €")
print(f"    Écart                  : {abs(bs_put - mc_put):.6f} €  ({abs(bs_put - mc_put)/bs_put*100:.4f}%)")
print(f"\n  Vol. implicite retrouvée : {iv_call*100:.4f}%  (on avait mis {sigma*100:.2f}%)")
print("\n" + "="*55)
print("\nConclusions :")
print("  - Black-Scholes est EXACT mais repose sur des hypothèses simplistes")
print("  - Monte Carlo est FLEXIBLE mais nécessite beaucoup de simulations")
print("  - L'erreur MC diminue en 1/√N (doubler la précision = 4x plus de calculs)")
print("  - La vol. implicite montre que le marché 'voit' des risques que BS ignore")